In [1]:
!pip install datasets

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset
import numpy as np

print("Downloading Amazon Reviews Dataset")
train_dataset = load_dataset("amazon_polarity", split="train[:20000]")
test_dataset = load_dataset("amazon_polarity", split="test[:5000]")

# Extract texts and labels
# Labels: 0 = Negative, 1 = Positive
X_train_texts = train_dataset['content']
y_train = train_dataset['label']
X_test_texts = test_dataset['content']
y_test = test_dataset['label']

In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 150

print("Tokenizing text...")
# Fit tokenizer on training data
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_texts)

# Convert strings to sequences of integers
X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
X_test_seq = tokenizer.texts_to_sequences(X_test_texts)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='pre', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='pre', truncating='post')

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train_pad, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test_pad, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data ready. Training on: {device}")

Tokenizing text...
Data ready. Training on: cpu


In [4]:
class GRU_Sentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(GRU_Sentiment, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.gru = nn.GRU(input_size=embed_dim,
                          hidden_size=hidden_dim,
                          batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        out, hidden = self.gru(embedded)
        final_hidden = hidden.squeeze(0)

        output = self.fc(final_hidden)
        return output.squeeze(1)

In [5]:
EMBED_DIM = 64
HIDDEN_DIM = 64
EPOCHS = 5

model = GRU_Sentiment(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n--- Starting Training ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss, total_correct = 0, 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        rounded_preds = torch.round(torch.sigmoid(predictions))
        total_correct += (rounded_preds == batch_y).sum().item()

    avg_loss = total_loss / len(train_loader)
    avg_acc = total_correct / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc*100:.2f}%")


--- Starting Training ---
Epoch 1/5 | Train Loss: 0.5952 | Train Acc: 66.54%
Epoch 2/5 | Train Loss: 0.4356 | Train Acc: 80.25%
Epoch 3/5 | Train Loss: 0.3454 | Train Acc: 85.50%
Epoch 4/5 | Train Loss: 0.2831 | Train Acc: 88.69%
Epoch 5/5 | Train Loss: 0.2312 | Train Acc: 91.15%


In [6]:
model.eval()
test_loss, test_correct = 0, 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)
        test_loss += loss.item()

        rounded_preds = torch.round(torch.sigmoid(predictions))
        test_correct += (rounded_preds == batch_y).sum().item()

final_test_loss = test_loss / len(test_loader)
final_test_acc = test_correct / len(test_loader.dataset)

print("\n--- Final Results ---")
print(f"Test Loss: {final_test_loss:.4f} | Test Accuracy: {final_test_acc*100:.2f}%")


--- Final Results ---
Test Loss: 0.3782 | Test Accuracy: 83.98%
